# BEV Fusion Exploration (Camera + LiDAR)

In [1]:
# Core imports for data loading + plotting.
from __future__ import annotations

import json
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Consistent plotting defaults make visual comparisons easier across cells.
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["image.cmap"] = "viridis"

# __file__ is not available in notebooks; discover repo root by walking up.
PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src" / "paths.py").exists():
        PROJECT_ROOT = candidate
        break

# Load paths.py from the exact file path to avoid stale/shadowed imports.
paths_file = PROJECT_ROOT / "src" / "paths.py"
spec = importlib.util.spec_from_file_location("project_paths", paths_file)
paths = importlib.util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
spec.loader.exec_module(paths)

ZOD_MOE_DATA = paths.ZOD_MOE_DATA
OUTPUTS_DIR = paths.OUTPUTS_DIR
ORIGINAL_DATA = paths.ZOD_DINO_DATA / "train2017"

print(f"Loaded paths from: {paths_file}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"ORIGINAL_DATA: {ORIGINAL_DATA}")
print(f"OUTPUTS_DIR: {OUTPUTS_DIR}")

first_frame_dir = ORIGINAL_DATA / "000000"
print(f"first_frame_dir: {first_frame_dir}")




Loaded paths from: /home/edgelab/multimodal-MoE/src/paths.py
PROJECT_ROOT: /home/edgelab/multimodal-MoE
ORIGINAL_DATA: /home/edgelab/zod_dino_data/train2017
OUTPUTS_DIR: /home/edgelab/multimodal-MoE/outputs
first_frame_dir: /home/edgelab/zod_dino_data/train2017/000000


In [2]:
#helpers
def _read_json(path: Path):
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"Could not read JSON: {path}\n{e}")
        return None


def _iso_to_dt(s: str) -> datetime:
    # Z-suffixed timestamp helper.
    # iso format: 2021-01-01T00:00:00.000000Z
    # datetime format: 2021-01-01 00:00:00+00:00
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


def find_keyframe_timestamp(frame_dir: Path, info_dict: dict) -> datetime:
    """
    Find the keyframe timestamp from the info.json file.
    """
    return _iso_to_dt(info_dict["keyframe_time"])


def nearest_lidar_filename(frame_dir: Path) -> str | None:
    """
    Input: Keyframe Directory Path Object
    Output: LiDAR Filename (lidar_velodyne) with timestamp closest to keyframe_time
    We read the info.json file to get the keyframe_time. 
    Instead of using the full LiDAR window ~ 11-12 .npy files, we just use the closest one. 
    """
    info = _read_json(frame_dir / "info.json")
    lidar_dir = frame_dir / "lidar_velodyne"

    if not isinstance(info, dict) or "keyframe_time" not in info or not lidar_dir.exists():
        return None

    keyframe_timestamp = _iso_to_dt(info["keyframe_time"])
    best_name = None
    # initialize best absolute delta time to infinity
    best_abs_dt = float("inf")

    for p in lidar_dir.glob("*.npy"):
        # Filename pattern: <frame_id>_<car>_<ISO8601>.npy
        timestamp_str = p.stem.rsplit("_", 1)[-1]
        try:
            delta_time_s = (_iso_to_dt(timestamp_str) - keyframe_timestamp).total_seconds()
        except Exception:
            continue

        abs_dt = abs(delta_time_s)
        if abs_dt < best_abs_dt:
            best_abs_dt = abs_dt
            best_name = p.name

    return best_name

nearest_name_example = nearest_lidar_filename(first_frame_dir)
print("Nearest LiDAR filename for FRAME_DIR:", nearest_name_example)







Nearest LiDAR filename for FRAME_DIR: 000000_india_2021-04-19T10:23:10.416676Z.npy
